# Practice: convolutions and a CNN that reads digits
*AI from scratch, chapter 6* · lesson: https://ihor.sh/ai/06-convolutions.html

Write a convolution, check it against PyTorch, work out output sizes and parameter counts, then train a small
convolutional network on the 70,000 MNIST handwritten digits and look at the filters it learned.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).
A GPU helps (Colab: Runtime → Change runtime type → T4 GPU; Kaggle: Accelerator → GPU, Internet on).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Extract sliding patches with sliding_window_view(I, K.shape).', 'Multiply each patch by K and sum spatial axes: np.einsum("rcij,ij->rc", sliding_window_view(I, K.shape), K).'], 2: ['Compute floor((n + 2 * p - k) // s) + 1.'], 3: ['Compute conv_params = out_channels * (in_channels * k * k) + out_channels = 64 * (3 * 3 * 3) + 64.', 'Compute dense_params = in_features * out_features + out_features = (32 * 32 * 3) * (30 * 30 * 64) + (30 * 30 * 64).'], 4: ['Two blocks of nn.Conv2d(in, out, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2): 1 → 16 channels, then 16 → 32.', 'Each MaxPool2d(2) halves the size: 28 → 14 → 7. So the last layer is nn.Linear(32 * 7 * 7, 10) after nn.Flatten().']}
_ANSWERS = {1: 'cmV0dXJuIG5wLmVpbnN1bSgicmNpaixpai0+cmMiLCBzbGlkaW5nX3dpbmRvd192aWV3KEksIEsuc2hhcGUpLCBLKQ==', 2: 'cmV0dXJuIChuICsgMiAqIHAgLSBrKSAvLyBzICsgMQ==', 3: 'Y29udl9wYXJhbXMgPSA2NCAqIDMgKiAzICogMyArIDY0CmRlbnNlX3BhcmFtcyA9ICgzMiAqIDMyICogMykgKiAoMzAgKiAzMCAqIDY0KSArIDMwICogMzAgKiA2NA==', 4: 'bW9kZWwgPSBubi5TZXF1ZW50aWFsKAogICAgbm4uQ29udjJkKDEsIDE2LCAzLCBwYWRkaW5nPTEpLCBubi5SZUxVKCksIG5uLk1heFBvb2wyZCgyKSwgICAgICMgMjjDlzI4IOKGkiAxNMOXMTQKICAgIG5uLkNvbnYyZCgxNiwgMzIsIDMsIHBhZGRpbmc9MSksIG5uLlJlTFUoKSwgbm4uTWF4UG9vbDJkKDIpLCAgICAjIDE0w5cxNCDihpIgN8OXNwogICAgbm4uRmxhdHRlbigpLCBubi5MaW5lYXIoMzIgKiA3ICogNywgMTApKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## 1. A convolution without loops
`sliding_window_view(I, (k, k))` gives every k×k patch of an image (shape (H−k+1, W−k+1, k, k)). Write
`conv2d(I, K)`: multiply each patch by the filter and sum (`np.einsum("rcij,ij->rc", ...)` or `(patches * K).sum(...)`).
PyTorch's `F.conv2d` does the same ("valid" mode, no flipping).

In [ ]:
def conv2d(I, K):
    """Compute 2D valid cross-correlation (convolution without flipping) of image I with kernel K.
    
    Args:
        I: 2D NumPy array representing image of shape (H, W).
        K: 2D NumPy array representing filter kernel of shape (k, k).
    Returns:
        NumPy array of shape (H - k + 1, W - k + 1).
    """
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

I, K = np.random.default_rng(0).random((10, 12)), np.random.default_rng(1).standard_normal((3, 3))
ref = F.conv2d(torch.tensor(I)[None, None], torch.tensor(K)[None, None])[0, 0].numpy()

In [ ]:
check(1, lambda: np.allclose(conv2d(I, K), ref), 
      "Your conv2d(I, K) implementation must match PyTorch's F.conv2d output.")

## 2. Output size
Write `out_size(n, k, p=0, s=1)`: the output width of a convolution on an n-wide input with kernel k, padding p,
stride s: $\lfloor (n + 2p - k)/s \rfloor + 1$. PyTorch's `nn.Conv2d` checks you.

In [ ]:
def out_size(n, k, p=0, s=1):
    """Calculate the output spatial dimension for a 1D/2D convolution.
    
    Formula: floor((n + 2*p - k) / s) + 1
    
    Args:
        n: Input spatial dimension (height or width).
        k: Kernel filter size.
        p: Symmetric padding applied to both sides (default 0).
        s: Convolution stride step (default 1).
    Returns:
        int: Output spatial dimension.
    """
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

cases = [(28, 3, 0, 1), (28, 3, 1, 1), (28, 5, 2, 2), (224, 7, 3, 2)]
torch_sizes = [nn.Conv2d(1, 1, k, stride=s, padding=p)(torch.zeros(1, 1, n, n)).shape[-1] for n, k, p, s in cases]

In [ ]:
check(2, lambda: [out_size(*c) for c in cases] == torch_sizes, 
      f"out_size results must match PyTorch's output sizes: {torch_sizes}")

## 3. Why convolutions: parameter counts
Set `conv_params`: the parameters of `nn.Conv2d(3, 64, 3)` (64 filters over 3 colour channels, plus 64 biases),
and `dense_params`: a fully connected layer mapping a 32×32×3 image to a 30×30×64 output. Count by formula, then
confirm the first with PyTorch (`sum(p.numel() for p in layer.parameters())`).

In [ ]:
conv_params = dense_params = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(conv_params, dense_params)

In [ ]:
check(3, lambda: conv_params == sum(p.numel() for p in nn.Conv2d(3, 64, 3).parameters()), 
      "nn.Conv2d(3, 64, 3) must have 1,792 total parameters.")
check(3, lambda: dense_params > 10_000 * conv_params, 
      "An equivalent unshared dense layer requires over 10,000x more parameters (> 170M params).")

## 4. Train a CNN on MNIST
Two conv blocks (conv 3×3 with padding 1 → ReLU → 2×2 max pool with `MaxPool2d(2)`), then a linear layer to 10 scores:
`Conv2d(1, 16, 3, padding=1)`, `ReLU()`, `MaxPool2d(2)`, `Conv2d(16, 32, 3, padding=1)`, `Flatten()`, `Linear(32 * 7 * 7, 10)`.
Build it as `nn.Sequential` in `model`. The training loop below runs 2 epochs with Adam.

In [ ]:
from torchvision import datasets, transforms
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("data", train=True, download=True, transform=tf)
test_ds = datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=512)

torch.manual_seed(0)
model = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...

def evaluate():
    model.eval(); right = 0
    with torch.no_grad():
        for x, y in test_dl:
            right += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
    return right / len(test_ds)

test_acc = None
if model is not None:
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(2):
        model.train()
        for x, y in train_dl:
            loss = F.cross_entropy(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        test_acc = evaluate()
        print(f"epoch {epoch + 1}: test accuracy {test_acc:.4f}")

In [ ]:
check(4, lambda: test_acc > 0.98, 
      "The two-layer CNN should reach > 98% test accuracy on MNIST.")
check(4, lambda: sum(p.numel() for p in model.parameters()) < 25_000, 
      "The entire CNN architecture should contain fewer than 25,000 parameters.")

## 5. What did the first layer learn?
Plot the 16 first-layer filters (`model[0].weight`, shape (16, 1, 3, 3)) as small grey images. Many look like the
lesson's edge detectors: light on one side, dark on the other.

In [ ]:
if test_acc is not None:
    w = model[0].weight.detach().cpu()[:, 0]
    fig, ax = plt.subplots(2, 8, figsize=(10, 3))
    for a, k in zip(ax.flat, w):
        a.imshow(k, cmap="gray"); a.axis("off")
    plt.show()

## Next
Back to the path: https://ihor.sh/ai/ · fast.ai lesson 1 trains an image classifier on photos you collect
yourself (https://course.fast.ai/Lessons/lesson1.html).